### Stage 4: Model Evaluation

Evaluates the fine-tuned Pegasus model using ROUGE metrics on the test split

In [1]:
import os
os.chdir('../')
print('Working dirctory:', os.getcwd())

Working dirctory: c:\Users\HP\Text-Summarizer


### 1. Config Entity

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    model_path: Path
    tokenizer_path: Path
    ingested_test_dir: Path
    input_column: str
    target_column: str
    batch_size: int
    max_generate_length: int
    num_beams: int
    metric_file_name: str

### 2. Configuration Manager



In [3]:
from src.textsummarizer.utils.common import read_yaml
from src.textsummarizer.utils.common import ConfigBox

CONFIG_FILE_PATH = Path('config/config.yaml')
PARAMS_FILE_PATH = Path('params.yaml')

class ConfigurationManager:
    def __init__(self):
        self.config = read_yaml(CONFIG_FILE_PATH)
        self.params = read_yaml(PARAMS_FILE_PATH)

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        cfg = self.config.artifacts.model_evaluation
        trainer_cfg = self.config.artifacts.model_trainer
        ing = self.config.artifacts.data_ingestion
        params = self.params.model_evaluation
        os.makedirs(cfg.root_dir, exist_ok=True)
        return ModelEvaluationConfig(
            root_dir=Path(cfg.root_dir),
            model_path=Path(trainer_cfg.root_dir) / 'pegasus-samsum-model',
            tokenizer_path=Path(trainer_cfg.root_dir) / 'tokenizer',
            ingested_test_dir=Path(ing.ingested_terst_dir),
            input_column=params.input_column,
            target_column=params.target_column,
            batch_size=params.batch_size,
            max_generate_length=params.max_generate_length,
            num_beams=params.num_beams,
            metric_file_name=cfg.metric_file_name,
        )

### 3. Model Evaluation Component


In [7]:
import torch
import pandas as pd
from tqdm import tqdm
from datasets import load_from_disk
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import evaluate

class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f'Using device: {self.device}')

    def generate_batch_sized_chunks(self, dataset, batch_size):
        """Yield successive batch_size chunks from the dataset"""
        for i in range(0, len(dataset), batch_size):
            yield dataset[i: i + batch_size]

    def calculate_metric_on_test_ds(self, dataset, metric, model, tokenizer):
        article_batches = list(
            self.generate_batch_sized_chunks(
                dataset[self.config,input_columns], self.config.batch_size
            )
        )
        target_batches = list(
            self.generate_batch_sized_chunks(
                dataset[self.config.target_column], self.config.batch_size
            )
        )
        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)):
            inputs = tokenizer(
                article=batch,
                max_length=1024,
                truncation=True,
                padding='max_length',
                return_tensors='pt'
            )

            summaries = model.generate(
                input_ids = inputs['input_ids'].to(self.device),
                attention_mask=inputs['attention_mask'].to(self.device),
                length_penalty=0.8,
                num_beams=self.config.num_beams,
                max_length=self.config.max_generate_length,
            )

            decode_summaries = [
                tokenizer.decode(
                    s, skip_special_tokens=True, clean_up_tokenization_spaces=True
                )
                for s in summaries
            ]

            metric.add_batch(
                predictions=decoded_summaries,
                referneces=target_batch,
            )

        return metric.compute()

    def evaluate(self):
        tokenizer = AutoTokenizer.from_pretrained(str(self.config.tokenizer_path))
        model = AutoModelForSeq2SeqLM.from_pretrained(str(self.config.model_path))
        model.to(self.device)
        model.eval()

        test_ds = load_from_disk(str(self.config.ingested_test_dir))
        rouge_metric = evaluate.load('rouge')

        print('Running evaluation on test set...')
        score = self.calculate_metric_on_test_ds(
            test_ds, rouge_metric, model, tokenizer
        )

        rouge_names = ['rouge1', 'rouge2', 'rougeL', 'rougeLsum']
        rouge_dict = {rn: round(score[rn], 4) for rn in rouge_names}

        df = pd.DataFrame(rouge_dict, index=['pegasus'])
        df.to_csv(str(self.config.root_dir / self.config.metric_file_name), index=False)

        print('\n=== ROUGE Scores ===')
        print(df.to_string())
        print(f'\nMetrics saved to: {self.config.root_dir / self.config.metric_file_name}')
        return df



ModuleNotFoundError: No module named 'evaluate'

### Run Pipeline

In [ ]:
try:
    config_manager = ConfigurationManager()
    eval_config    = config_manager.get_model_evaluation_config()
    evaluator      = ModelEvaluation(config=eval_config)
    results_df     = evaluator.evaluate()
except Exception as e:
    raise e

### 5. Visualize

In [ ]:
import matplotlib.pyplot as plt

ax = results_df.T.plot(
    kind='bar', figsize=(8, 4), legend=False,
    color=['#4C72B0', '#DD8452', '#55A868', '#C44E52']
)
ax.set_title('ROUGE Scores — Pegasus Fine-tuned')
ax.set_ylabel('F-measure')
ax.set_ylim(0, 1)
ax.set_xticklabels(results_df.columns, rotation=0)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.4f}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('artifacts/model_evaluation/rouge_scores.png', dpi=150)
plt.show()

### Quick Inference Check

In [ ]:
from transformers import pipeline
from pathlib import Path

summarizer = pipeline(
    'summarization',
    model=str(eval_config.model_path),
    tokenizer=str(eval_config.tokenizer_path),
)

# Use any sample from the test set
from datasets import load_from_disk
test_ds = load_from_disk(str(eval_config.ingested_test_dir))
sample  = test_ds[0]

print('--- INPUT ---')
print(sample[eval_config.input_column])
print('\n--- REFERENCE SUMMARY ---')
print(sample[eval_config.target_column])
print('\n--- GENERATED SUMMARY ---')
result = summarizer(sample[eval_config.input_column],
                    max_length=eval_config.max_generate_length,
                    min_length=30)
print(result[0]['summary_text'])